# Ejercicio 2 — Feature Extraction con 6 clases: Imagenette (subconjunto) con EfficientNetB0

**Curso:** IS056B — Ingeniería del Conocimiento | **Sesión 09-A:** Transfer Learning — Feature Extraction

**Objetivo:** Aplicar Feature Extraction a un problema de **clasificación multiclase (6 clases)**, incluyendo análisis exploratorio, Data Augmentation, métricas de evaluación y visualización de feature maps.

**Dataset:** Imagenette (versión 160px), un subconjunto de 10 clases de ImageNet. Se utilizan 6 de ellas:

| # | Clase | Tipo |
|---|---|---|
| 0 | Tenca (pez) | Animal |
| 1 | Springer spaniel inglés | Animal |
| 2 | Iglesia | Construcción |
| 3 | Corno francés | Objeto |
| 4 | Camión de basura | Vehículo |
| 5 | Paracaídas | Objeto |

**Modelo base:** EfficientNetB0 pre-entrenado en ImageNet (congelado).

> Nota: las clases de Imagenette provienen de ImageNet, por lo que el backbone ya las vio durante su preentrenamiento.

## Configuración del entorno

In [ ]:
import subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'], capture_output=True, text=True)
print("GPU detectada:", gpu_info.stdout.strip() or "No hay GPU - activa en Runtime > Change runtime type > T4 GPU")

In [ ]:
import os, tarfile
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
SEED = 42
BATCH_SIZE = 32
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TensorFlow {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

---
## 2.1. Descargar el dataset, seleccionar 6 clases y análisis exploratorio

Imagenette organiza las imágenes en carpetas con códigos de WordNet. Se traducen a nombres legibles y se cargan únicamente las 6 carpetas elegidas.

In [ ]:
DATA_DIR = os.path.abspath('datos')
os.makedirs(DATA_DIR, exist_ok=True)

def descargar(nombre, urls):
    ultimo_error = None
    for url in urls:
        try:
            return tf.keras.utils.get_file(nombre, origin=url, cache_dir=DATA_DIR,
                                           cache_subdir='descargas')
        except Exception as e:
            ultimo_error = e
            print(f"No se pudo descargar desde {url}: {e}")
    raise RuntimeError(f"No fue posible descargar {nombre}") from ultimo_error

def raiz_con_carpetas(base, carpetas):
    for r, dirs, _ in os.walk(base):
        if all(c in dirs for c in carpetas):
            return r
    raise FileNotFoundError(f"No se encontraron las carpetas {carpetas} en {base}")

ruta_tgz = descargar('imagenette2-160.tgz', [
    'https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz'])

destino = os.path.join(DATA_DIR, 'imagenette')
if not os.path.isdir(destino):
    with tarfile.open(ruta_tgz) as t:
        try:
            t.extractall(destino, filter='data')
        except TypeError:
            t.extractall(destino)

raiz = raiz_con_carpetas(destino, ['train', 'val'])
train_dir = os.path.join(raiz, 'train')
val_dir   = os.path.join(raiz, 'val')
print("Carpeta de entrenamiento:", train_dir)
print("Carpeta de validación:   ", val_dir)

In [ ]:
WNID_TO_NAME = {
    'n01440764': 'Tenca',
    'n02102040': 'Springer spaniel',
    'n03028079': 'Iglesia',
    'n03394916': 'Corno francés',
    'n03417042': 'Camión de basura',
    'n03888257': 'Paracaídas',
}
SELECTED_WNIDS = list(WNID_TO_NAME.keys())
CLASS_NAMES_2 = [WNID_TO_NAME[w] for w in SELECTED_WNIDS]
NUM_CLASSES_2 = len(CLASS_NAMES_2)
IMG_SIZE_2 = (224, 224)

common = dict(image_size=IMG_SIZE_2, batch_size=BATCH_SIZE,
              label_mode='categorical', class_names=SELECTED_WNIDS)

train_raw_2 = tf.keras.utils.image_dataset_from_directory(train_dir, seed=SEED, **common)
val_raw_2   = tf.keras.utils.image_dataset_from_directory(val_dir, shuffle=False, **common)

print(f"\nClases ({NUM_CLASSES_2}): {CLASS_NAMES_2}")

In [ ]:
# Distribución de clases
def contar(ds):
    c = Counter(os.path.basename(os.path.dirname(p)) for p in ds.file_paths)
    return [c[w] for w in SELECTED_WNIDS]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES_2))
for ax, ds, title_str in [(axes[0], train_raw_2, 'Train'), (axes[1], val_raw_2, 'Validación')]:
    counts = contar(ds)
    bars = ax.bar(CLASS_NAMES_2, counts, color=colors, edgecolor='gray')
    ax.set_title(f'Distribución — {title_str}', fontweight='bold')
    ax.set_ylabel('Cantidad')
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(count), ha='center', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Imagenette (6 clases) — Distribución de clases', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Muestras por clase (4 por clase)
N_COLS = 4
paths_by_class = {w: [] for w in SELECTED_WNIDS}
for p in train_raw_2.file_paths:
    w = os.path.basename(os.path.dirname(p))
    if len(paths_by_class[w]) < N_COLS:
        paths_by_class[w].append(p)

fig, axes = plt.subplots(NUM_CLASSES_2, N_COLS, figsize=(12, 2.6 * NUM_CLASSES_2))
for r, w in enumerate(SELECTED_WNIDS):
    for c in range(N_COLS):
        ax = axes[r, c]
        ax.imshow(tf.keras.utils.load_img(paths_by_class[w][c]))
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel(CLASS_NAMES_2[r], fontsize=11, fontweight='bold',
                          rotation=0, labelpad=70, va='center')
plt.suptitle('Muestras por clase — Imagenette (6 clases)', fontsize=14, fontweight='bold', y=1.0)
plt.tight_layout(); plt.show()

---
## 2.2. Data Augmentation

El augmentation se incluye como parte del modelo y **solo se activa durante el entrenamiento**. EfficientNetB0 incorpora su propia normalización de píxeles, por lo que recibe las imágenes en el rango [0, 255].

In [ ]:
from tensorflow.keras import layers, models

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name='augmentation')

AUTOTUNE = tf.data.AUTOTUNE
train_ds_2 = train_raw_2.prefetch(AUTOTUNE)
val_ds_2   = val_raw_2.prefetch(AUTOTUNE)

for imgs, labels in train_ds_2.take(1):
    print(f"Batch: {imgs.shape} | Labels: {labels.shape}")
    print(f"Rango de píxeles: [{imgs.numpy().min():.0f}, {imgs.numpy().max():.0f}]")

In [ ]:
# Visualizar el augmentation sobre una misma imagen
ruta = train_raw_2.file_paths[0]
sample = tf.image.resize(tf.keras.utils.img_to_array(tf.keras.utils.load_img(ruta)), IMG_SIZE_2)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes[0, 0].imshow(sample.numpy().astype('uint8')); axes[0, 0].set_title('Original', fontweight='bold'); axes[0, 0].axis('off')
for i in range(1, 10):
    ax = axes[i // 5, i % 5]
    aug = data_augmentation(tf.expand_dims(sample, 0), training=True)
    ax.imshow(aug[0].numpy().astype('uint8'))
    ax.set_title(f'Augmented #{i}', fontsize=10); ax.axis('off')
plt.suptitle(f'Data Augmentation — {WNID_TO_NAME[os.path.basename(os.path.dirname(ruta))]}',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 2.3. Construir el modelo con EfficientNetB0 (6 clases)

La capa de salida es `Dense(6, softmax)`, una neurona por clase.

In [ ]:
base_model_2 = tf.keras.applications.EfficientNetB0(
    weights='imagenet', include_top=False, input_shape=(*IMG_SIZE_2, 3))
base_model_2.trainable = False

inputs_2 = layers.Input(shape=(*IMG_SIZE_2, 3))
x = data_augmentation(inputs_2)                   # solo activo en entrenamiento
x = base_model_2(x, training=False)               # BN en modo inferencia
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs_2 = layers.Dense(NUM_CLASSES_2, activation='softmax')(x)

model_2 = models.Model(inputs_2, outputs_2, name='imagenette6_efficientnetb0_FE')

total = model_2.count_params()
trainable = sum(tf.size(w).numpy() for w in model_2.trainable_weights)
print(f"EfficientNetB0 - Feature Extraction ({NUM_CLASSES_2} clases)")
print(f"Total: {total:,} | Entrenables: {trainable:,} | Congelados: {total-trainable:,}")

---
## 2.4. Entrenar con callbacks

In [ ]:
model_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_2 = model_2.fit(
    train_ds_2,
    validation_data=val_ds_2,
    epochs=15,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2,
                                             min_lr=1e-6, verbose=1),
    ]
)
print(f"\nMejor val_accuracy: {max(history_2.history['val_accuracy']):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history_2.history['accuracy']) + 1)

axes[0].plot(ep, history_2.history['accuracy'], 'o-', label='Train', color='#4CAF50')
axes[0].plot(ep, history_2.history['val_accuracy'], 's-', label='Val', color='#E91E63')
axes[0].set_title('Accuracy — Imagenette 6 clases (EfficientNetB0 FE)', fontweight='bold')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history_2.history['loss'], 'o-', label='Train', color='#4CAF50')
axes[1].plot(ep, history_2.history['val_loss'], 's-', label='Val', color='#E91E63')
axes[1].set_title('Loss — Imagenette 6 clases (EfficientNetB0 FE)', fontweight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 2.5. Evaluación detallada: Classification Report y Confusion Matrix (6×6)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in val_ds_2])
y_pred = np.argmax(model_2.predict(val_ds_2, verbose=0), axis=1)

print("Classification Report - Imagenette 6 clases (EfficientNetB0 FE)")
print("=" * 70)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES_2, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES_2,
            yticklabels=CLASS_NAMES_2, ax=ax, linewidths=0.5, linecolor='white',
            annot_kws={'size': 12, 'fontweight': 'bold'})
ax.set_xlabel('Predicción', fontsize=13, fontweight='bold')
ax.set_ylabel('Real', fontsize=13, fontweight='bold')
ax.set_title('Confusion Matrix — Imagenette 6 clases (EfficientNetB0 FE)', fontsize=14, fontweight='bold')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

print("\nPrincipales confusiones (3 o más errores):")
found = False
for i in range(NUM_CLASSES_2):
    for j in range(NUM_CLASSES_2):
        if i != j and cm[i, j] >= 3:
            print(f"   {CLASS_NAMES_2[i]} -> {CLASS_NAMES_2[j]}: {cm[i, j]} errores"); found = True
if not found:
    print("   Ninguna confusión llega a 3 errores.")

---
## 2.6. Predicciones visuales (validación)

In [ ]:
viz_raw = tf.keras.utils.image_dataset_from_directory(
    val_dir, shuffle=True, seed=SEED, image_size=IMG_SIZE_2, batch_size=10,
    label_mode='categorical', class_names=SELECTED_WNIDS)
imgs, labels = next(iter(viz_raw))
probs = model_2.predict(imgs, verbose=0)

fig, axes = plt.subplots(2, 5, figsize=(16, 7.5))
for i, ax in enumerate(axes.ravel()):
    pred_idx = int(np.argmax(probs[i])); true_idx = int(np.argmax(labels[i].numpy()))
    ax.imshow(imgs[i].numpy().astype('uint8'))
    ax.set_title(f"Pred: {CLASS_NAMES_2[pred_idx]} ({probs[i].max():.0%})", fontsize=10,
                 color='green' if pred_idx == true_idx else 'red', fontweight='bold')
    ax.set_xlabel(f"Real: {CLASS_NAMES_2[true_idx]}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Predicciones — EfficientNetB0 FE (6 clases)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 2.7. Visualización de Feature Maps

Se extraen las activaciones de 4 capas del backbone, de profundidad creciente.

In [ ]:
layer_names = ['block2a_expand_conv', 'block4a_expand_conv', 'block6a_expand_conv', 'top_conv']
feature_models = [
    models.Model(inputs=base_model_2.input, outputs=base_model_2.get_layer(n).output)
    for n in layer_names
]

ruta = val_raw_2.file_paths[len(val_raw_2.file_paths) // 2]
sample_img = tf.keras.utils.load_img(ruta)
sample_in = tf.expand_dims(tf.image.resize(tf.keras.utils.img_to_array(sample_img), IMG_SIZE_2), 0)

fig, axes = plt.subplots(len(layer_names), 9, figsize=(18, 2.5 * len(layer_names)))
for row, (fm, lname) in enumerate(zip(feature_models, layer_names)):
    features = fm.predict(sample_in, verbose=0)
    axes[row, 0].imshow(sample_img)
    axes[row, 0].set_title('Original' if row == 0 else '', fontsize=9)
    axes[row, 0].set_ylabel(lname.replace('_', '\n'), fontsize=9, fontweight='bold',
                            rotation=0, labelpad=60, va='center')
    axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    for col in range(1, 9):
        ax = axes[row, col]
        ax.imshow(features[0, :, :, col - 1], cmap='viridis'); ax.axis('off')
        if row == 0:
            ax.set_title(f'Ch {col}', fontsize=9)
plt.suptitle(f'Feature Maps por Capa — {WNID_TO_NAME[os.path.basename(os.path.dirname(ruta))]}',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

Las capas tempranas responden a bordes y texturas, las intermedias a patrones más complejos y las profundas a abstracciones de alto nivel.

---
## Conclusión del Ejercicio 2

In [ ]:
best_val = max(history_2.history['val_accuracy'])
resumen = pd.DataFrame({
    'Métrica': ['Modelo base', 'Dataset', 'Clases', 'Imágenes train / val',
                'Parámetros entrenables', 'Épocas ejecutadas', 'Mejor Val Accuracy', 'Augmentation'],
    'Valor': ['EfficientNetB0 (ImageNet, congelado)', 'Imagenette 160px (subconjunto)',
              f"{NUM_CLASSES_2}: {', '.join(CLASS_NAMES_2)}",
              f"{len(train_raw_2.file_paths):,} / {len(val_raw_2.file_paths):,}",
              f"{trainable:,} (cabeza)", len(history_2.history['accuracy']),
              f"{best_val:.4f}", 'Flip + Rotation + Zoom + Contrast']
})
print(resumen.to_string(index=False))